# 24. qml4var Supervised vs Unsupervised (Multi-Strike Price Curves, SPSA)


In [ ]:
import sys
from time import perf_counter

import torch

sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats

from finance import (
    ak_bk_from_complex_coefficients,
    complex_fourier_coefficients,
    fourier_price_v_t0,
)
from qml4var.adam import adam_optimizer_loop
from qml4var.architectures import (
    hardware_efficient_ansatz,
    init_weights,
    normalize_data,
)
from qml4var.data_utils import empirical_cdf
from qml4var.workflows import (
    qdml_loss_workflow,
    unsupervised_qdml_loss_workflow,
    workflow_for_cdf,
    workflow_for_pdf,
)

try:
    from tqdm.auto import tqdm
    TQDM_AVAILABLE = True
except Exception:
    TQDM_AVAILABLE = False

    class _NoTqdm:
        def __init__(self, total=None, desc=None, unit=None, disable=False):
            self.total = total
            self.n = 0
            self.disable = disable
            if not self.disable:
                print(f"{desc or 'Progress'} | total={total} {unit or ''}")

        def update(self, n=1):
            self.n += n

        def set_postfix(self, *args, **kwargs):
            return None

        def close(self):
            return None

    def tqdm(*args, **kwargs):
        return _NoTqdm(*args, **kwargs)


## 1. Global Setup

In [ ]:
# Black-Scholes parameters
S0 = 100.0
r = 0.10
sigma = 0.25
T = 1.0
STRIKES = [90, 100, 110]

# Domains
TRAIN_INTERVAL = (-2.0 * np.pi, 2.0 * np.pi)  # workflow domain
DATA_INTERVAL = (-1.0 * np.pi, 1.0 * np.pi)   # data interval after rescaling
ENCODING_INTERVAL = (-1.0 * np.pi, 1.0 * np.pi)
FEATURES_NUMBER = 1

# Fourier pricing setup
K_TERMS = 12
GRID_POINTS_FOURIER = 1024

# Optional: use Dask client
DASK_CLIENT = None

# Torch device (GPU if available, else CPU)
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("TORCH_DEVICE:", TORCH_DEVICE)

# Normalization from training domain [-2pi, 2pi] into encoding [-pi, pi]
base_frecuency, shift_feature = normalize_data(
    [TRAIN_INTERVAL[0]] * FEATURES_NUMBER,
    [TRAIN_INTERVAL[1]] * FEATURES_NUMBER,
    [ENCODING_INTERVAL[0]] * FEATURES_NUMBER,
    [ENCODING_INTERVAL[1]] * FEATURES_NUMBER,
)

print("base_frecuency:", base_frecuency)
print("shift_feature:", shift_feature)


## 2. Independent Configurations by Mode

You can edit each mode independently, without touching training functions.


In [3]:
MODE_CONFIGS = {
    "supervised": {
        "learning_rate": 0.005,
        "epochs": 50,
        "loss_weights": [0.9, 0.1],
        "train_sizes": [250, 500, 1000],
        "test_points": 100,
        "repetitions": 1,
        "beta1": 0.9,
        "beta2": 0.999,
        "tolerance": 1.0e-8,
        "n_counts_tolerance": 25,
        "print_step": 50,
        "batch_size": None,
        "n_qubits_by_feature": 3,
        "n_layers": 3,
        "debug_price_stats": True,
        "show_epoch_progress": True,
        "epoch_progress_leave": False,
        "spsa": {"c": 1.0e-3, "seed": 12345},
    },
    "unsupervised": {
        "learning_rate": 0.1,
        "epochs": 50,
        "loss_weights": [0.2, 0.8],
        "train_sizes": [1000, 2500, 5000],
        "test_points": 1000,
        "repetitions": 1,
        "beta1": 0.9,
        "beta2": 0.999,
        "tolerance": 1.0e-8,
        "n_counts_tolerance": 25,
        "print_step": 50,
        "batch_size": None,
        "empirical_shift": -0.5,
        "n_qubits_by_feature": 2,
        "n_layers": 2,
        "debug_price_stats": True,
        "show_epoch_progress": True,
        "epoch_progress_leave": False,
        "spsa": {"c": 1.0e-3, "seed": 12345},
    },
}

MODE_CONFIGS


{'supervised': {'learning_rate': 0.005,
  'epochs': 50,
  'loss_weights': [0.9, 0.1],
  'train_sizes': [250, 500, 1000],
  'test_points': 100,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 50,
  'batch_size': None,
  'n_qubits_by_feature': 3,
  'n_layers': 3,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False,
  'spsa': {'c': 0.001, 'seed': 12345}},
 'unsupervised': {'learning_rate': 0.1,
  'epochs': 50,
  'loss_weights': [0.2, 0.8],
  'train_sizes': [1000, 2500, 5000],
  'test_points': 1000,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 50,
  'batch_size': None,
  'empirical_shift': -0.5,
  'n_qubits_by_feature': 2,
  'n_layers': 2,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False,
  'spsa': {'c': 0.001, 'seed': 12345}}}

## 3. Helper Functions

In [ ]:
def trapz_compat(y, x):
    fn = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return fn(y=y, x=x)


def bs_put_price(S0_, K_, r_, sigma_, T_):
    d1 = (np.log(S0_ / K_) + (r_ + 0.5 * sigma_**2) * T_) / (sigma_ * np.sqrt(T_))
    d2 = d1 - sigma_ * np.sqrt(T_)
    return K_ * np.exp(-r_ * T_) * stats.norm.cdf(-d2) - S0_ * stats.norm.cdf(-d1)


def build_mode_artifacts(cfg):
    pqc_cfg = {
        "features_number": FEATURES_NUMBER,
        "n_qubits_by_feature": int(cfg["n_qubits_by_feature"]),
        "n_layers": int(cfg["n_layers"]),
        "base_frecuency": list(base_frecuency),
        "shift_feature": list(shift_feature),
    }
    circuit_fn, weights_names, features_names = hardware_efficient_ansatz(**pqc_cfg)

    workflow_cfg = {
        "circuit_fn": circuit_fn,
        "weights_names": weights_names,
        "features_names": features_names,
        "torch_device": TORCH_DEVICE,
        "minval": [TRAIN_INTERVAL[0]],
        "maxval": [TRAIN_INTERVAL[1]],
        "points": 150,
    }
    return {
        "pqc_cfg": pqc_cfg,
        "weights_names": weights_names,
        "workflow_cfg": workflow_cfg,
    }


def simulate_black_scholes_data_rescaled(S0_, r_, T_, sigma_, K_, n_points, seed):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal(n_points)
    s_t = S0_ * np.exp((r_ - 0.5 * sigma_**2) * T_ + sigma_ * np.sqrt(T_) * z)
    x_t = np.log(s_t / K_)

    x_min_raw = float(np.min(x_t))
    x_max_raw = float(np.max(x_t))
    if x_max_raw <= x_min_raw:
        x_max_raw = x_min_raw + 1.0e-8

    # Rescale to [-pi, pi]
    u_t = 2.0 * np.pi * (x_t - x_min_raw) / (x_max_raw - x_min_raw) - np.pi
    return x_t.reshape(-1, 1), u_t.reshape(-1, 1), x_min_raw, x_max_raw


def inverse_rescaling_u_to_xt(u_values, x_min_raw, x_max_raw):
    u_values = np.asarray(u_values)
    return x_min_raw + (u_values + np.pi) * (x_max_raw - x_min_raw) / (2.0 * np.pi)


def batch_generator(X, Y, batch_size):
    return [(X[i : i + batch_size], Y[i : i + batch_size]) for i in range(0, len(X), batch_size)]


def spsa_gradient(weights, data_x, data_y, loss_fn, spsa_cfg=None):
    """
    SPSA gradient estimator using only 2 loss evaluations per step.

    Parameters
    ----------
    weights : array-like
        Current trainable weights.
    data_x : np.array
        Batch features.
    data_y : np.array
        Batch targets.
    loss_fn : callable
        Function with signature: loss_fn(weights, x, y) -> scalar loss.
    spsa_cfg : dict or None
        Optional config with keys:
            c : float (default 1.0e-3), perturbation magnitude.
            seed : int or None, reproducible random generator seed.
            rng : np.random.Generator (optional), takes precedence over seed.

    Returns
    -------
    grad : list
        Estimated gradient list with same length as weights.
    """
    cfg = {} if spsa_cfg is None else dict(spsa_cfg)
    c = float(cfg.get("c", 1.0e-3))

    rng = cfg.get("rng", None)
    if rng is None:
        seed = cfg.get("seed", None)
        if seed is None:
            rng = np.random.default_rng()
        else:
            if not hasattr(spsa_gradient, "_rng_cache"):
                spsa_gradient._rng_cache = {}
            if seed not in spsa_gradient._rng_cache:
                spsa_gradient._rng_cache[seed] = np.random.default_rng(seed)
            rng = spsa_gradient._rng_cache[seed]

    w = np.asarray(weights, dtype=float)
    delta = rng.choice(np.array([-1.0, 1.0]), size=w.shape)

    loss_plus = float(loss_fn(w + c * delta, data_x, data_y))
    loss_minus = float(loss_fn(w - c * delta, data_x, data_y))

    grad = (loss_plus - loss_minus) / (2.0 * c * delta)
    return grad.tolist()


def run_supervised(u_train, cfg, artifacts, progress_desc=None):
    workflow_cfg = artifacts["workflow_cfg"]
    weights_names = artifacts["weights_names"]

    y_train = empirical_cdf(u_train).reshape((-1, 1)) - 0.5
    batch_size = cfg["batch_size"] if cfg["batch_size"] is not None else len(u_train)
    batches = batch_generator(u_train, y_train, batch_size)

    def training_loss(w_):
        return qdml_loss_workflow(
            w_, u_train, y_train, dask_client=DASK_CLIENT, loss_weights=cfg["loss_weights"], **workflow_cfg
        )

    def loss_for_grad(w_, x_, y_):
        return qdml_loss_workflow(
            w_, x_, y_, dask_client=DASK_CLIENT, loss_weights=cfg["loss_weights"], **workflow_cfg
        )

    spsa_cfg = cfg.get("spsa", {"c": 1.0e-3, "seed": 12345})

    def gradient_fn(w_, x_, y_):
        return spsa_gradient(w_, x_, y_, loss_for_grad, spsa_cfg=spsa_cfg)

    optimizer_cfg = {
        "epochs": cfg["epochs"],
        "learning_rate": cfg["learning_rate"],
        "beta1": cfg["beta1"],
        "beta2": cfg["beta2"],
        "tolerance": cfg["tolerance"],
        "n_counts_tolerance": cfg["n_counts_tolerance"],
        "print_step": cfg["print_step"],
        "progress_bar": bool(cfg.get("show_epoch_progress", False)),
        "progress_desc": progress_desc if progress_desc is not None else "supervised epochs",
        "progress_leave": bool(cfg.get("epoch_progress_leave", False)),
    }

    initial_weights = init_weights(weights_names)
    return adam_optimizer_loop(
        weights_dict=initial_weights,
        loss_function=training_loss,
        metric_function=None,
        gradient_function=gradient_fn,
        batch_generator=batches,
        initial_time=0,
        **optimizer_cfg,
    )


def run_unsupervised(u_train, cfg, artifacts, progress_desc=None):
    workflow_cfg = artifacts["workflow_cfg"]
    weights_names = artifacts["weights_names"]

    dummy_y = np.zeros((u_train.shape[0], 1))
    batch_size = cfg["batch_size"] if cfg["batch_size"] is not None else len(u_train)
    batches = batch_generator(u_train, dummy_y, batch_size)

    def training_loss(w_):
        return unsupervised_qdml_loss_workflow(
            w_,
            u_train,
            dask_client=DASK_CLIENT,
            empirical_shift=cfg["empirical_shift"],
            loss_weights=cfg["loss_weights"],
            **workflow_cfg,
        )

    def loss_for_grad(w_, x_, y_):
        return unsupervised_qdml_loss_workflow(
            w_,
            x_,
            dask_client=DASK_CLIENT,
            empirical_shift=cfg["empirical_shift"],
            loss_weights=cfg["loss_weights"],
            **workflow_cfg,
        )

    spsa_cfg = cfg.get("spsa", {"c": 1.0e-3, "seed": 12345})

    def gradient_fn(w_, x_, y_):
        return spsa_gradient(w_, x_, y_, loss_for_grad, spsa_cfg=spsa_cfg)

    optimizer_cfg = {
        "epochs": cfg["epochs"],
        "learning_rate": cfg["learning_rate"],
        "beta1": cfg["beta1"],
        "beta2": cfg["beta2"],
        "tolerance": cfg["tolerance"],
        "n_counts_tolerance": cfg["n_counts_tolerance"],
        "print_step": cfg["print_step"],
        "progress_bar": bool(cfg.get("show_epoch_progress", False)),
        "progress_desc": progress_desc if progress_desc is not None else "unsupervised epochs",
        "progress_leave": bool(cfg.get("epoch_progress_leave", False)),
    }

    initial_weights = init_weights(weights_names)
    return adam_optimizer_loop(
        weights_dict=initial_weights,
        loss_function=training_loss,
        metric_function=None,
        gradient_function=gradient_fn,
        batch_generator=batches,
        initial_time=0,
        **optimizer_cfg,
    )


def estimate_price_from_trained_pqc(
        weights,
        artifacts,
        K_,
        x_min_raw,
        x_max_raw,
        k_terms=12,
        grid_points=1024,
        debug=False,
        debug_label=""):
    workflow_cfg = artifacts["workflow_cfg"]
    u_grid = np.linspace(TRAIN_INTERVAL[0], TRAIN_INTERVAL[1], grid_points).reshape((-1, 1))

    pdf_raw = workflow_for_pdf(weights, u_grid, dask_client=DASK_CLIENT, **workflow_cfg)["y_predict_pdf"].reshape(-1)

    pdf_pred = np.nan_to_num(pdf_raw, nan=0.0, posinf=0.0, neginf=0.0)
    pdf_pred = np.clip(pdf_pred, 0.0, None)

    area = trapz_compat(pdf_pred, u_grid[:, 0])

    if debug:
        print(
            f"[PRICE DEBUG] {debug_label} "
            f"pdf_min={np.min(pdf_pred):.6e} pdf_max={np.max(pdf_pred):.6e} area={area:.6e}"
        )

    if (not np.isfinite(area)) or np.isclose(area, 0.0):
        print(
            f"[WARN price] {debug_label} invalid area. "
            f"pdf_min={np.min(pdf_pred):.6e} pdf_max={np.max(pdf_pred):.6e} area={area}"
        )
        return np.nan

    pdf_pred = pdf_pred / area

    if np.allclose(pdf_pred, 0.0):
        print(f"[WARN price] {debug_label} normalized pdf collapsed to zeros")
        return np.nan

    k_vals, c_k = complex_fourier_coefficients(
        x_domain=u_grid[:, 0],
        y_predict=pdf_pred,
        k_values=k_terms,
        interval=TRAIN_INTERVAL,
    )
    _, A_k_f, B_k_f = ak_bk_from_complex_coefficients(k_vals, c_k, k_max=k_terms)

    x_raw_grid = inverse_rescaling_u_to_xt(u_grid[:, 0], x_min_raw, x_max_raw)
    payoff = np.maximum(K_ * (1.0 - np.exp(x_raw_grid)), 0.0)

    L = TRAIN_INTERVAL[1] - TRAIN_INTERVAL[0]
    z = (u_grid[:, 0] - TRAIN_INTERVAL[0]) / L
    C_k = np.zeros(k_terms + 1, dtype=complex)
    D_k = np.zeros(k_terms + 1, dtype=complex)
    for k in range(k_terms + 1):
        angle = 2.0 * np.pi * k * z
        C_k[k] = (2.0 / L) * trapz_compat(payoff * np.cos(angle), u_grid[:, 0])
        if k == 0:
            D_k[k] = 0.0
        else:
            D_k[k] = (2.0 / L) * trapz_compat(payoff * np.sin(angle), u_grid[:, 0])

    v_t0 = fourier_price_v_t0(
        a=TRAIN_INTERVAL[0],
        b=TRAIN_INTERVAL[1],
        risk_free_rate=r,
        delta_t=T,
        a_k_f=A_k_f,
        b_k_f=B_k_f,
        c_k_payoff=C_k,
        d_k_payoff=D_k,
    )
    return float(np.real(v_t0))

## 4. Debug

Use these toggles before launching the full loop.


In [5]:
# Optional quick sanity mode
DEBUG_FAST_RUN = False

# More frequent progress logs (helps confirm the process is alive)
VERBOSE_PROGRESS = True

if VERBOSE_PROGRESS:
    for cfg in MODE_CONFIGS.values():
        cfg["print_step"] = min(int(cfg.get("print_step", 50)), 10)
        cfg["print_step"] = max(cfg["print_step"], 1)

if DEBUG_FAST_RUN:
    # tiny run to verify everything end-to-end before expensive execution
    for cfg in MODE_CONFIGS.values():
        cfg["epochs"] = min(int(cfg["epochs"]), 2)
        cfg["repetitions"] = 1
        cfg["train_sizes"] = [int(cfg["train_sizes"][0])]

MODE_CONFIGS


{'supervised': {'learning_rate': 0.005,
  'epochs': 50,
  'loss_weights': [0.9, 0.1],
  'train_sizes': [250, 500, 1000],
  'test_points': 100,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 10,
  'batch_size': None,
  'n_qubits_by_feature': 3,
  'n_layers': 3,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False,
  'spsa': {'c': 0.001, 'seed': 12345}},
 'unsupervised': {'learning_rate': 0.1,
  'epochs': 50,
  'loss_weights': [0.2, 0.8],
  'train_sizes': [1000, 2500, 5000],
  'test_points': 1000,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 10,
  'batch_size': None,
  'empirical_shift': -0.5,
  'n_qubits_by_feature': 2,
  'n_layers': 2,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False,
  'spsa': {'c': 0.001, 'seed': 12345}}}

In [6]:
def expected_total_runs(mode_configs, strikes):
    return sum(len(cfg["train_sizes"]) * len(strikes) * int(cfg["repetitions"]) for cfg in mode_configs.values())


def format_eta(seconds):
    if seconds is None or (not np.isfinite(seconds)):
        return "n/a"
    seconds = int(max(0, round(seconds)))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def eta_from_partial(results_df, mode_configs, strikes):
    total_runs = expected_total_runs(mode_configs, strikes)
    done_runs = len(results_df)
    if done_runs == 0:
        return {"total_runs": total_runs, "done_runs": 0, "avg_sec_per_run": None, "eta_hours": None}
    avg_sec = float(results_df["seconds"].mean())
    pending = max(total_runs - done_runs, 0)
    eta_hours = (pending * avg_sec) / 3600.0
    return {
        "total_runs": total_runs,
        "done_runs": done_runs,
        "avg_sec_per_run": avg_sec,
        "eta_hours": eta_hours,
    }


USE_TQDM = True

# Usage after (or during) the main loop:
# eta_from_partial(results_df, MODE_CONFIGS, STRIKES)


## 4. Run Experiment 

In [7]:
import os

os.environ["QML4VAR_PROFILE_TRAINING"] = "1"
os.environ["QML4VAR_PROFILE_EPOCHS"] = "1"
os.environ["QML4VAR_PROFILE_ONCE"] = "1"
os.environ["QML4VAR_PROFILE_LABEL"] = "first_case"

os.environ["QML4VAR_PROFILE_GRAD"] = "1"
os.environ["QML4VAR_PROFILE_GRAD_FIRST_ONLY"] = "1"


In [8]:
results = []
base_seed = 2026

mode_artifacts = {}
for mode_name, cfg in MODE_CONFIGS.items():
    artifacts = build_mode_artifacts(cfg)
    mode_artifacts[mode_name] = artifacts
    print(
        f"mode={mode_name} -> qubits_by_feature={cfg['n_qubits_by_feature']} "
        f"layers={cfg['n_layers']} total_weights={len(artifacts['weights_names'])}"
    )

total_runs = expected_total_runs(MODE_CONFIGS, STRIKES)
print(f"\nPlanned training runs: {total_runs}")

progress = tqdm(
    total=total_runs,
    desc="Training runs",
    unit="run",
    disable=not (USE_TQDM and TQDM_AVAILABLE),
)

run_counter = 0
global_t0 = perf_counter()

for mode_name, cfg in MODE_CONFIGS.items():
    artifacts = mode_artifacts[mode_name]
    print(f"\n=== Running mode: {mode_name} ===")
    for K_ in STRIKES:
        theor_price = bs_put_price(S0, K_, r, sigma, T)
        for n_train in cfg["train_sizes"]:
            n_test = int(cfg["test_points"])
            for rep in range(cfg["repetitions"]):
                seed = base_seed + 100000 * rep + 1000 * int(K_) + n_train
                n_total = n_train + n_test

                x_raw_all, u_all, x_min_raw, x_max_raw = simulate_black_scholes_data_rescaled(
                    S0_=S0, r_=r, T_=T, sigma_=sigma, K_=K_, n_points=n_total, seed=seed
                )
                u_train = u_all[:n_train]
                u_test = u_all[n_train:]

                t0 = perf_counter()
                epoch_desc = f"{mode_name} K={K_} n={n_train} rep={rep + 1}"
                if mode_name == "supervised":
                    weights = run_supervised(u_train, cfg, artifacts, progress_desc=epoch_desc)
                else:
                    weights = run_unsupervised(u_train, cfg, artifacts, progress_desc=epoch_desc)
                t1 = perf_counter()

                # Optional test metric (CDF MSE wrt empirical CDF on test split)
                y_test = empirical_cdf(u_test).reshape((-1, 1)) - 0.5
                y_pred_test = workflow_for_cdf(
                    weights, u_test, dask_client=DASK_CLIENT, **artifacts["workflow_cfg"]
                )["y_predict_cdf"].reshape((-1, 1))
                test_mse = float(np.mean((y_pred_test - y_test) ** 2))

                debug_label = f"mode={mode_name} K={K_} n_train={n_train} rep={rep + 1}"
                est_price = estimate_price_from_trained_pqc(
                    weights=weights,
                    artifacts=artifacts,
                    K_=K_,
                    x_min_raw=x_min_raw,
                    x_max_raw=x_max_raw,
                    k_terms=K_TERMS,
                    grid_points=GRID_POINTS_FOURIER,
                    debug=bool(cfg.get("debug_price_stats", False)),
                    debug_label=debug_label,
                )

                run_seconds = float(t1 - t0)
                results.append({
                    "mode": mode_name,
                    "strike": K_,
                    "dataset_size": n_train,
                    "test_points": n_test,
                    "rep": rep + 1,
                    "estimated_price": est_price,
                    "theoretical_price": float(theor_price),
                    "test_mse": test_mse,
                    "seconds": run_seconds,
                    "n_qubits_by_feature": int(cfg["n_qubits_by_feature"]),
                    "n_layers": int(cfg["n_layers"]),
                })

                run_counter += 1
                elapsed = perf_counter() - global_t0
                avg_per_run = elapsed / max(run_counter, 1)
                eta_seconds = (total_runs - run_counter) * avg_per_run

                progress.update(1)
                progress.set_postfix({
                    "mode": mode_name,
                    "K": K_,
                    "n": n_train,
                    "rep": rep + 1,
                    "run_s": f"{run_seconds:.1f}",
                    "eta": format_eta(eta_seconds),
                })

                print(
                    f"mode={mode_name} K={K_} n_train={n_train} rep={rep + 1} "
                    f"price={est_price:.6f} run_s={run_seconds:.2f} ETA={format_eta(eta_seconds)}"
                )

progress.close()

total_elapsed = perf_counter() - global_t0
print(f"\nCompleted {run_counter}/{total_runs} runs in {format_eta(total_elapsed)}")

results_df = pd.DataFrame(results)
results_df.head()


mode=supervised -> qubits_by_feature=3 layers=3 total_weights=9
mode=unsupervised -> qubits_by_feature=2 layers=2 total_weights=4

Planned training runs: 18


Training runs:   0%|          | 0/18 [00:00<?, ?run/s]


=== Running mode: supervised ===


	 MSE at t=0: None
	 Iteracion: 0. Loss: 0.4120696922567323
[PROFILE first_case] epoch=0 batches=1 grad_s=46.492 update_s=0.000 loss_s=22.156 metric_s=0.000 misc_s=0.003 total_s=68.651


	 MSE at t=10: None
	 Iteracion: 10. Loss: 0.3991523737499886


...

	 MSE at t=40: None
	 Iteracion: 40. Loss: -0.03591094290039671


Maximum number of iterations achieved.


Training runs: 100%|██████████| 18/18 [84:47:22<00:00, 16957.91s/run, mode=unsupervised, K=110, n=5000, rep=1, run_s=23406.9, eta=00:00:00]  

[PRICE DEBUG] mode=unsupervised K=110 n_train=5000 rep=1 pdf_min=0.000000e+00 pdf_max=7.979429e-02 area=3.182294e-01
mode=unsupervised K=110 n_train=5000 rep=1 price=18.011773 run_s=23406.90 ETA=00:00:00

Completed 18/18 runs in 48:47:58


,mode,strike,dataset_size,test_points,rep,estimated_price,theoretical_price,test_mse,seconds,n_qubits_by_feature,n_layers
0,supervised,90,250,100,1,2.345166,2.598827,0.248862,3473.205165,3,3
1,supervised,90,500,100,1,8.358181,2.598827,0.198684,5978.162508,3,3
2,supervised,90,1000,100,1,2.898649,2.598827,0.278117,9793.726631,3,3
3,supervised,100,250,100,1,5.359322,5.459533,0.008577,3188.928444,3,3
4,supervised,100,500,100,1,4.527054,5.459533,0.151514,5229.687042,3,3


## 5. Summaries

In [9]:
summary_df = results_df.groupby(["mode", "strike", "dataset_size"], as_index=False).agg(
    mean_price=("estimated_price", "mean"),
    std_price=("estimated_price", "std"),
    mean_test_mse=("test_mse", "mean"),
    mean_seconds=("seconds", "mean"),
)
summary_df


,mode,strike,dataset_size,mean_price,std_price,mean_test_mse,mean_seconds
0,supervised,90,250,2.345166,NaN,0.248862,3473.205165
1,supervised,90,500,8.358181,NaN,0.198684,5978.162508
2,supervised,90,1000,2.898649,NaN,0.278117,9793.726631
3,supervised,100,250,5.359322,NaN,0.008577,3188.928444
4,supervised,100,500,4.527054,NaN,0.151514,5229.687042
5,supervised,100,1000,7.038879,NaN,0.014888,9303.615239
6,supervised,110,250,4.771606,NaN,0.106877,3201.267920
7,supervised,110,500,2.268819,NaN,0.068153,5290.643215
8,supervised,110,1000,13.413018,NaN,0.107219,10191.889583
9,unsupervised,90,1000,9.698853,NaN,0.044983,3883.911918


## 6. Price Curves by Dataset Size


In [ ]:
colors = ["#377eb8", "#ff7f00", "#4daf4a"]

for mode_name, cfg in MODE_CONFIGS.items():
    dataset_sizes = cfg["train_sizes"]
    df_mode = results_df[results_df["mode"] == mode_name].copy()

    plt.figure(figsize=(10, 7))

    for K_, color in zip(STRIKES, colors, strict=False):
        df_k = df_mode[df_mode["strike"] == K_]

        means = []
        stds = []
        for n_ in dataset_sizes:
            vals = df_k[df_k["dataset_size"] == n_]["estimated_price"].values
            means.append(np.nanmean(vals) if vals.size > 0 else np.nan)
            stds.append(np.nanstd(vals, ddof=1) if vals.size > 1 else 0.0)

        plt.errorbar(
            dataset_sizes,
            means,
            yerr=stds,
            fmt="-o",
            color=color,
            label=f"K = {K_}",
            capsize=5,
            markersize=8,
            elinewidth=1.5,
        )

        # Theoretical line for each strike
        p_theo = bs_put_price(S0, K_, r, sigma, T)
        plt.axhline(p_theo, color=color, linestyle="--", linewidth=0.9, alpha=0.9)

        # Outliers by IQR
        for n_ in dataset_sizes:
            vals = df_k[df_k["dataset_size"] == n_]["estimated_price"].dropna().values
            if vals.size < 4:
                continue
            q1, q3 = np.percentile(vals, [25, 75])
            iqr = q3 - q1
            low = q1 - 1.5 * iqr
            high = q3 + 1.5 * iqr
            outliers = vals[(vals < low) | (vals > high)]
            if outliers.size > 0:
                plt.scatter(
                    np.full(outliers.shape[0], n_),
                    outliers,
                    marker="x",
                    color=color,
                    s=45,
                    alpha=0.95,
                    label=f"Outliers K={K_}" if n_ == dataset_sizes[0] else None,
                )

    plt.title(f"Estimated prices by dataset size | mode={mode_name} | test_points={cfg['test_points']}")
    plt.xlabel("Size of the dataset (n_train)")
    plt.ylabel("Price")
    plt.xticks(dataset_sizes)
    plt.grid(alpha=0.25, axis="y")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()
